# 1. Basics

This notebook introduces the `HyperGraph` class in `hypergraph.py` and shows how to construct a hypergraph from data and compute its basic structural properties.

## What is a hypergraph?

A **hypergraph** consists of:
- A set of **nodes** $V = \{v_1, \ldots, v_N\}$
- A set of **hyperedges** $E = \{e_1, \ldots, e_M\}$, where each hyperedge $e_j \subseteq V$ can contain an arbitrary number of nodes

Unlike ordinary graphs — where each edge connects exactly two nodes — a hyperedge can connect any number of nodes simultaneously. This makes hypergraphs well-suited for representing group interactions (e.g., co-authorship, group conversations, multi-protein complexes).

Two fundamental quantities are:
- **Node degree** $k_i$: the number of hyperedges to which node $v_i$ belongs
- **Hyperedge size** $s_j$: the number of nodes in hyperedge $e_j$

---

If you use this code, please cite:

[Kazuki Nakajima, Kazuyuki Shudo, Naoki Masuda. Randomizing Hypergraphs Preserving Degree Correlation and Local Clustering. *IEEE TNSE*. vol. 9, pp. 1139-1153 (2021).](https://doi.org/10.1109/TNSE.2021.3133380)

Ensure the version of Python is >=3.10

In [1]:
import sys
print(sys.executable)

/Users/knakajima/.pyenv/versions/3.10.15/bin/python3


## Getting Started

Import `hypergraph.py`.

In [2]:
import hypergraph

## Toy Example

Construct a small hypergraph to explore the API.
Represent hyperedges as a list of lists, where each list is a hyperedge, and pass them to `add_hyperedges_from`.

**Note:** `HyperGraph` allows multiple hyperedges and self-loops. For example, `[0, 1, 2]` appears twice in `E` below, and node `5` appears twice in `[3, 4, 5, 5]`. While the original hypergraph data you work with may not contain such cases, randomized instances can in general.

In [3]:
H = hypergraph.HyperGraph()

In [4]:
E = [[0, 1, 2], [0, 1, 2], [2, 3, 4], [3, 4, 5, 5]]
H.add_hyperedges_from(E)

### Basic info

Print a summary of the hypergraph.

In [5]:
H.print_info()

Number of nodes: 6
Number of hyperedges: 4
Average degree of node: 2.1666666666666665
Maximum degree of node: 3
Frequency distribution of node degree: {2: 5, 3: 1}
Average size of hyperedge: 3.25
Maximum size of hyperedge: 4
Frequency distribution of hyperedge size: {3: 3, 4: 1}
Hypergraph is connected: True



### Nodes and hyperedges

Retrieve the set of nodes, the list of hyperedges, and the incidence relationships between them.

In [6]:
H.nodes()

{0, 1, 2, 3, 4, 5}

In [7]:
H.hyperedges()

[[0, 1, 2], [0, 1, 2], [2, 3, 4], [3, 4, 5, 5]]

In [8]:
H.hyperedges_incident_with_node(3)

[2, 3]

In [9]:
H.nodes_incident_with_hyperedge(0)

[0, 1, 2]

### Degree and size

The **degree** of a node $v_i$ is the number of hyperedges to which $v_i$ belongs.  
The **size** of a hyperedge $e_j$ is the number of nodes in $e_j$.

In [10]:
H.node_degree()

{0: 2, 1: 2, 2: 3, 3: 2, 4: 2, 5: 2}

In [11]:
H.node_degree(3)

2

In [12]:
H.hyperedge_size()

{0: 3, 1: 3, 2: 3, 3: 4}

In [13]:
H.hyperedge_size(0)

3

### Higher-order structural properties

**Joint degree distribution** $P(k, k')$ measures the degree correlation between nodes that share at least one hyperedge. Letting $m(k, k')$ denote the number of hyperedges shared by nodes of degree $k$ and degree $k'$:

$$P(k, k') = \frac{2\, m(k, k')}{\sum_{j=1}^{M} s_j(s_j - 1)}$$

This quantity is one of the key properties that the hyper $dK$-series with $d_v = 2$ aims to preserve. See [[Nakajima et al., IEEE TNSE (2021)]](https://doi.org/10.1109/TNSE.2021.3133380) for details.

In [14]:
H.joint_node_degree_distribution()

{(2, 2): 0.6, (2, 3): 0.2, (3, 2): 0.2, (3, 3): 0.0}

**Redundancy coefficient** $r_i$ quantifies local clustering around a node in the hypergraph — analogous to the clustering coefficient in ordinary graphs. It measures the fraction of pairs of hyperedges to which $v_i$ belongs such that at least one other node also belongs to both hyperedges:

$$r_i = \frac{2}{k_i(k_i-1)} \sum_{j < j'} B_{i,j}\, B_{i,j'}\, \mathbf{1}[|\Gamma_{jj'}| > 0]$$

where $\Gamma_{jj'}$ is the set of nodes other than $v_i$ that belong to both $e_j$ and $e_{j'}$, and $B$ is the node–hyperedge incidence matrix. $r_i = 0$ if $k_i \leq 1$.

In [15]:
H.node_redundancy_coefficient()

{0: 1.0, 1: 1.0, 2: 0.3333333333333333, 3: 1.0, 4: 1.0, 5: 0.0}

**Degree-dependent redundancy coefficient** $r(k)$ is the average of $r_i$ over all nodes with degree $k$:

$$r(k) = \frac{1}{n(k)} \sum_{i:\, k_i = k} r_i$$

where $n(k)$ is the number of nodes with degree $k$. This is the quantity that the hyper $dK$-series with $d_v = 2.5$ aims to preserve.

In [16]:
H.degree_dependent_node_redundancy_coefficient()

{2: 0.8, 3: 0.3333333333333333}

**Shortest path length distribution** between nodes. Two nodes are considered connected if they share at least one hyperedge (distance = 1), or are reachable through a sequence of shared hyperedges.

In [17]:
H.node_shortest_path_length_distribution()

{1: 16, 2: 10, 3: 4}

### Conversion to HyperNetX

The `HyperGraph` object can be converted to a [`hypernetx.Hypergraph`](https://github.com/pnnl/HyperNetX) for visualization and further analysis.

**Note:** Since our `HyperGraph` allows multiple hyperedges and self-loops, the incidence matrix entry for a node–hyperedge pair reflects the number of times the node appears in that hyperedge.

In [18]:
H_hnx = H.convert_to_hnx_hypergraph()
H_hnx.dataframe

,edges,nodes,weight,misc_properties
0,0,0,1,{}
1,1,0,1,{}
2,0,1,1,{}
3,1,1,1,{}
4,0,2,1,{}
5,1,2,1,{}
6,2,2,1,{}
7,2,3,1,{}
8,3,3,1,{}
9,2,4,1,{}


## Real Data

Read hypergraph data from the `data/` directory using `read_hypergraph`. The function expects two files: `<name>_nverts.txt` and `<name>_hyperedges.txt`. See the README for the file format.

### `example-hypergraph` (5 nodes, 5 hyperedges)

In [19]:
H = hypergraph.read_hypergraph("example-hypergraph")

In [20]:
H.print_info()

Number of nodes: 5
Number of hyperedges: 5
Average degree of node: 3.2
Maximum degree of node: 5
Frequency distribution of node degree: {4: 2, 5: 1, 2: 1, 1: 1}
Average size of hyperedge: 3.2
Maximum size of hyperedge: 5
Frequency distribution of hyperedge size: {2: 2, 3: 1, 4: 1, 5: 1}
Hypergraph is connected: True



### `syn1000` (1,000 nodes, 4,996 hyperedges)

A synthetic hypergraph with a heterogeneous degree distribution.

In [21]:
H = hypergraph.read_hypergraph("syn1000")

In [22]:
H.print_info()

Number of nodes: 1000
Number of hyperedges: 4996
Average degree of node: 15.992
Maximum degree of node: 86
Frequency distribution of node degree: {67: 1, 46: 1, 39: 1, 81: 1, 86: 1, 47: 1, 13: 86, 29: 4, 34: 2, 24: 10, 26: 3, 38: 3, 30: 1, 25: 14, 40: 2, 11: 67, 19: 45, 43: 1, 35: 1, 20: 47, 18: 73, 9: 41, 23: 20, 17: 70, 31: 1, 51: 1, 16: 115, 15: 114, 21: 27, 22: 28, 14: 87, 12: 67, 33: 1, 27: 3, 10: 35, 7: 5, 8: 17, 6: 2, 5: 1}
Average size of hyperedge: 3.200960768614892
Maximum size of hyperedge: 5
Frequency distribution of hyperedge size: {2: 1996, 3: 1000, 4: 1000, 5: 1000}
Hypergraph is connected: True

